In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Generic Overpass/GeoJSON -> single-layer GeoPackage in EPSG:2056.

Robust to:
- invalid geometries (tries pyogrio on_invalid='ignore', fallback to fiona)
- annoying columns (dict/list/object -> JSON strings; drops known bad fields)
- CRS missing (assumes EPSG:4326)
- optional geometry repair for polygons (make_valid if available, else buffer(0))
"""

from __future__ import annotations

import json
from pathlib import Path
from typing import Iterable

import geopandas as gpd
import pandas as pd


# ----------------------------
# Helpers
# ----------------------------
def _to_json_str(x):
    if isinstance(x, (dict, list, tuple, set)):
        try:
            return json.dumps(x, ensure_ascii=False)
        except Exception:
            return str(x)
    return x


def _safe_attributes(
    gdf: gpd.GeoDataFrame,
    drop_cols: Iterable[str] = ("fixme", "nodes", "members", "bounds", "center", "meta", "tags", "properties"),
) -> gpd.GeoDataFrame:
    """Drop troublesome columns and serialize nested objects to JSON strings."""
    out = gdf.copy()

    # Drop known problematic columns if present
    cols_to_drop = [c for c in out.columns if c in set(drop_cols)]
    if cols_to_drop:
        out = out.drop(columns=cols_to_drop, errors="ignore")

    # Convert object columns to JSON-safe strings
    for c in out.columns:
        if c == "geometry":
            continue
        if out[c].dtype == "object":
            out[c] = out[c].map(_to_json_str).astype("string")

    # GeoPackage field-name length is OK, but keep it sane
    # (Optional: uncomment if you hit field name issues)
    # out.columns = [str(c)[:120] for c in out.columns]

    return out


def _repair_polygons_only(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Attempt to repair invalid polygon geometries.
    - Uses shapely.make_valid() if available
    - Else uses buffer(0) only on polygonal geometries
    Lines/points are left untouched.
    """
    out = gdf.copy()
    # Only operate on invalid geoms
    invalid = ~out.geometry.is_valid
    if invalid.sum() == 0:
        return out

    try:
        # Shapely >= 2
        from shapely import make_valid  # type: ignore

        out.loc[invalid, "geometry"] = out.loc[invalid, "geometry"].apply(make_valid)
        return out

    except Exception:
        # Fallback: buffer(0) ONLY for polygons (safe-ish)
        poly_mask = out.geometry.geom_type.isin(["Polygon", "MultiPolygon"]) & invalid
        if poly_mask.sum() > 0:
            out.loc[poly_mask, "geometry"] = out.loc[poly_mask, "geometry"].buffer(0)
        return out


# ----------------------------
# Main generic converter
# ----------------------------
def overpass_geojson_to_gpkg_single_layer(
    in_geojson: str | Path,
    out_gpkg: str | Path,
    layer_name: str = "outdoor_hard",
    target_epsg: int = 2056,
    try_repair_polygons: bool = True,
) -> gpd.GeoDataFrame:
    """
    Reads a GeoJSON exported from Overpass Turbo and writes a single-layer GPKG.
    Returns the cleaned GeoDataFrame (in target CRS).
    """
    in_path = Path(in_geojson)
    out_path = Path(out_gpkg)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    print(f"📂 Reading: {in_path}")

    # 1) Robust read
    try:
        gdf = gpd.read_file(in_path, on_invalid="ignore")  # pyogrio path
        print("   ✅ Read with pyogrio (on_invalid='ignore').")
    except Exception as e:
        print(f"   ⚠️ pyogrio failed ({type(e).__name__}: {e}) -> trying engine='fiona'...")
        gdf = gpd.read_file(in_path, engine="fiona")
        print("   ✅ Read with Fiona.")

    if gdf.empty:
        raise ValueError("Input file is empty after reading (or all geometries were invalid and ignored).")

    print(f"   Rows read: {len(gdf)}")
    print(f"   Columns: {list(gdf.columns)}")

    # 2) CRS handling (Overpass exports usually EPSG:4326)
    if gdf.crs is None:
        gdf = gdf.set_crs(epsg=4326)
        print("   ℹ️ CRS missing -> set to EPSG:4326.")

    # 3) Drop NaN/empty geometries
    before = len(gdf)
    gdf = gdf[gdf.geometry.notna()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    print(f"   🔍 Dropped {before - len(gdf)} NaN/empty geometries.")

    # 4) Optional polygon repair (only if still invalids remain)
    if try_repair_polygons:
        n_invalid_before = (~gdf.geometry.is_valid).sum()
        if n_invalid_before > 0:
            print(f"   🛠️ Attempting to repair {n_invalid_before} invalid geometries (polygons only)...")
            gdf = _repair_polygons_only(gdf)

    # 5) Final invalid drop (keep it clean)
    before = len(gdf)
    gdf = gdf[gdf.geometry.is_valid].copy()
    print(f"   🔍 Dropped {before - len(gdf)} geometries still invalid after repair.")

    if gdf.empty:
        raise ValueError("All features were removed as invalid/empty.")

    # 6) Attributes cleanup (fixme, nested tags, etc.)
    gdf = _safe_attributes(gdf)

    # 7) Reproject to target
    if gdf.crs.to_epsg() != target_epsg:
        print(f"   ↪ Reprojecting EPSG:{gdf.crs.to_epsg()} -> EPSG:{target_epsg}")
        gdf = gdf.to_crs(epsg=target_epsg)

    # 8) Write single layer
    # Overwrite existing file for reproducibility (optional)
    if out_path.exists():
        out_path.unlink()

    gdf.to_file(out_path, layer=layer_name, driver="GPKG")
    print(f"💾 Saved: {out_path} (layer='{layer_name}', EPSG:{target_epsg}) | rows={len(gdf)}")

    return gdf


# ----------------------------
# Example usage (your case)
# ----------------------------
if __name__ == "__main__":
    base = Path(".")
    in_geojson = base / "OSM" / "Gastronomy" / "F1_Gastronomy.geojson"
    out_gpkg = base / "OSM" / "Gastronomy" / "F1_Gastronomy.gpkg"

    overpass_geojson_to_gpkg_single_layer(
        in_geojson=in_geojson,
        out_gpkg=out_gpkg,
        layer_name="outdoor_hard",
        target_epsg=2056,
        try_repair_polygons=True,
    )
